In [ ]:
%%capture
import warnings
warnings.filterwarnings('ignore')
import calitp_data_analysis.magics

import speedmap_utils
from shared_utils import webmap_utils, catalog_utils, rt_utils
import pandas as pd
import datetime as dt
import numpy as np

from functools import cache

from calitp_data_analysis.gcs_geopandas import GCSGeoPandas
from calitp_data_analysis.gcs_pandas import GCSPandas
import scipy
import geopandas as gpd

@cache
def gcs_pandas():
    return GCSPandas()

@cache
def gcs_geopandas():
    return GCSGeoPandas()

catalog = catalog_utils.get_catalog("gtfs_analytics_data")

In [ ]:
SPEED_TRIPS_PATH = f"{catalog.speedmap_segments.dir}{catalog.speedmap_segments.stage4}"
SPEED_SEGS_PATH = f"{catalog.speedmap_segments.dir}{catalog.speedmap_segments.segment_timeofday}"

In [ ]:
from update_vars_index import ANALYSIS_DATE_LIST
analysis_date = ANALYSIS_DATE_LIST[0]

In [ ]:
analysis_date

In [ ]:
path = f"{SPEED_SEGS_PATH}_{analysis_date}.parquet"
speedmap_segs = gcs_geopandas().read_parquet(path, filters=[["time_of_day", "==", "All Day"]])  # aggregated

In [ ]:
speedmap_segs[speedmap_segs.analysis_name.isna()].name.unique() # TODO analysis_name guidance/add...

In [ ]:
msg = "no cols besides route_short_name, direction_id should be nan"
assert speedmap_segs.drop(columns=["route_short_name", "direction_id", "analysis_name"]).isna().any().any() == False, msg

In [ ]:
shs = gcs_geopandas().read_parquet(rt_utils.SHN_PATH)

In [ ]:
from calitp_data_analysis.geography_utils import CA_NAD83Albers_m

## Spatial Operations

In [ ]:
speedmap_segs = speedmap_segs.to_crs(CA_NAD83Albers_m)
shs = shs.to_crs(CA_NAD83Albers_m)

### add linear intersect meters with shs

In [ ]:
linear_intersect = speedmap_segs.copy().overlay(shs, how='intersection')
linear_intersect = linear_intersect.assign(approx_shs_intersect_meters = linear_intersect.geometry.map(lambda x: x.length))

In [ ]:
# linear_intersect.sample(5000).explore()

In [ ]:
linear_intersect = linear_intersect[['schedule_gtfs_dataset_key', 'segment_id', 'approx_shs_intersect_meters']].round(1)

In [ ]:
speedmap_segs = speedmap_segs.merge(linear_intersect, on = ['schedule_gtfs_dataset_key', 'segment_id'])

### Final sjoin

In [ ]:
shs_segs = gpd.sjoin(speedmap_segs, shs, how='inner', predicate='intersects')
shs_segs = speedmap_utils.prepare_segment_gdf(shs_segs).assign(analysis_date=analysis_date)

In [ ]:
speedmap_segs.shape

In [ ]:
shs_segs.shape

# Clean trip speeds, calculate delay based on p20 times

In [ ]:
df = shs_segs

In [ ]:
trip_speeds = gcs_pandas().read_parquet(f'{SPEED_TRIPS_PATH}_{analysis_date}.parquet')

In [ ]:
# https://www.geeksforgeeks.org/machine-learning/z-score-for-outlier-detection-python/

trip_speeds = trip_speeds[['segment_id', 'sec_elapsed', 'schedule_gtfs_dataset_key', 'speed_mph']]
trip_speeds = trip_speeds.dropna()

# trip_speeds['speed_mph_z_score'] = scipy.stats.zscore(trip_speeds.speed_mph)
trip_speeds['sec_elapsed_z_score'] = scipy.stats.zscore(trip_speeds.sec_elapsed)



In [ ]:
trip_speeds.head(5)

In [ ]:
# don't use z score for speeds

outliers = trip_speeds[(trip_speeds.speed_mph <= 0.1) | (trip_speeds.speed_mph > 80) | (trip_speeds.sec_elapsed_z_score.abs() >= 3)]
trip_speeds = trip_speeds[(trip_speeds.speed_mph >= 0.1) & (trip_speeds.speed_mph < 80) & (trip_speeds.sec_elapsed != 0)]
trip_speeds = trip_speeds[trip_speeds.sec_elapsed_z_score.abs() < 3]

In [ ]:
trip_speeds.sort_values('speed_mph')

In [ ]:
# outliers

In [ ]:
with_benchmark_times = (trip_speeds
    .groupby(['segment_id', 'schedule_gtfs_dataset_key'])
    .agg({"sec_elapsed": lambda x: np.quantile(x, q=.2), "speed_mph": lambda x: np.quantile(x, q=.8)})
    .rename(columns={'sec_elapsed': 'p20_seconds_benchmark', 'speed_mph': 'p80_mph_benchmark'})
    .reset_index()
    .merge(trip_speeds, on = ['segment_id', 'schedule_gtfs_dataset_key'])
)

In [ ]:
with_benchmark_times.head(3)

In [ ]:
with_benchmark_times = with_benchmark_times.assign(transit_delay_sec = (with_benchmark_times.sec_elapsed - with_benchmark_times.p20_seconds_benchmark).clip(lower=0))

In [ ]:
with_benchmark_times

In [ ]:
delay_by_segment = (with_benchmark_times
    .groupby(['segment_id', 'schedule_gtfs_dataset_key'])[['transit_delay_sec']]
    .sum()
    .rename(columns={'transit_delay_sec': 'total_transit_delay_sec'})
    .reset_index()
)
delay_by_segment = (delay_by_segment
                    .assign(transit_veh_hrs_delay = delay_by_segment.total_transit_delay_sec / 60**2)
                    .round(2)
)

In [ ]:
delay_by_segment

In [ ]:
shs_delay = df.merge(delay_by_segment, on = ['segment_id', 'schedule_gtfs_dataset_key']).sort_values('transit_veh_hrs_delay', ascending=True)

In [ ]:
more_than_6min = shs_delay[shs_delay.transit_veh_hrs_delay > .1]

In [ ]:
more_than_6min.shape

In [ ]:
more_than_6min.columns

In [ ]:
map_cols = ['analysis_name', 'transit_veh_hrs_delay', 'n_trips',
             'n_trips_sch',
            'route_short_name', 'geometry', 'time_of_day',
            'p20_mph', 'p50_mph', 'p80_mph',
            'Route', 'District', 'RouteType',
            'name'
]

In [ ]:
more_than_6min[map_cols].explore(column='transit_veh_hrs_delay', scheme='Quantiles')

2